# 02 — Deterministic Triage Baseline and Guardrails

This notebook validates the finalized deterministic claims-triage baseline,
its controlled tool adapter, and the guardrail that prevents an agent layer
from overriding authoritative eligibility routing.

In [2]:
# ============================================================
# Notebook Setup: Project Root and Imports
# ============================================================

from pathlib import Path
import sys

CURRENT_DIRECTORY = Path.cwd().resolve()

if (CURRENT_DIRECTORY / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY
elif (CURRENT_DIRECTORY.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the 'src' folder. "
        f"Current working directory: {CURRENT_DIRECTORY}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Notebook working directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)
print("Source folder found:", (PROJECT_ROOT / "src").exists())

Notebook working directory: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/notebooks
Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Source folder found: True


In [5]:
# ============================================================
# Notebook Setup: Load Development Runtime Data
# ============================================================

import pandas as pd

from src.data_loader import load_runtime_data

data = load_runtime_data()

print("Runtime datasets loaded:")
for dataset_name, dataframe in data.items():
    print(f"{dataset_name}: {dataframe.shape[0]:,} rows x {dataframe.shape[1]} columns")

assert "development_claims" in data
assert len(data["development_claims"]) == 165

print("\nDevelopment claims available:", len(data["development_claims"]))

Runtime datasets loaded:
plan_master: 3 rows x 16 columns
coverage_matrix: 18 rows x 11 columns
evidence_requirements: 9 rows x 7 columns
policy_rules: 19 rows x 20 columns
policy_lookup: 120 rows x 18 columns
prior_claims: 112 rows x 16 columns
evidence_bundles: 220 rows x 8 columns
evidence_documents: 283 rows x 14 columns
development_claims: 165 rows x 14 columns
risk_results: 4 rows x 19 columns

Development claims available: 165


In [6]:
from src.agent.response_guardrail import build_guarded_agent_response
from src.tools.deterministic_triage import run_deterministic_triage

print("Deterministic triage tool imported successfully.")
print("Response guardrail imported successfully.")

Deterministic triage tool imported successfully.
Response guardrail imported successfully.


In [7]:
# ============================================================
# Step 28: Response Guardrail Validation
# ============================================================

from src.agent.response_guardrail import build_guarded_agent_response
from src.tools.deterministic_triage import run_deterministic_triage

tool_result = run_deterministic_triage(
    data=data,
    claim_id="CLM-000204",
)

aligned_proposal = {
    "case_summary": (
        "The case was assessed using the deterministic policy workflow."
    ),
    "reviewer_note": (
        "Review the authoritative trace and policy-limit facts."
    ),
    "next_step_message": (
        "Route the case according to the authoritative decision."
    ),
}

override_attempt = {
    "triage_outcome": "PROCEED",
    "triggering_rule_id": "OUT-001",
    "precedence_stage": 6,
    "decision_reason": "The claim has been approved.",
    "case_summary": (
        "The deterministic rule trace identified an annual-limit issue."
    ),
    "unexpected_field": "This field should be ignored.",
}

aligned_result = build_guarded_agent_response(
    tool_result=tool_result,
    proposed_response=aligned_proposal,
)

blocked_result = build_guarded_agent_response(
    tool_result=tool_result,
    proposed_response=override_attempt,
)

assert (
    aligned_result["triage_outcome"]
    == tool_result["deterministic_decision"]["triage_outcome"]
)

assert (
    blocked_result["triage_outcome"]
    == tool_result["deterministic_decision"]["triage_outcome"]
)

assert (
    blocked_result["triggering_rule_id"]
    == tool_result["deterministic_decision"]["triggering_rule_id"]
)

assert (
    blocked_result["authority_guardrail"]["status"]
    == "OVERRIDE_BLOCKED"
)

guardrail_validation_df = pd.DataFrame(
    [
        {
            "scenario": "Aligned explanation",
            "final_outcome": aligned_result["triage_outcome"],
            "final_rule": aligned_result["triggering_rule_id"],
            "guardrail_status": aligned_result[
                "authority_guardrail"
            ]["status"],
            "conflicting_fields": aligned_result[
                "authority_guardrail"
            ]["conflicting_authority_fields"],
        },
        {
            "scenario": "Attempted override",
            "final_outcome": blocked_result["triage_outcome"],
            "final_rule": blocked_result["triggering_rule_id"],
            "guardrail_status": blocked_result[
                "authority_guardrail"
            ]["status"],
            "conflicting_fields": blocked_result[
                "authority_guardrail"
            ]["conflicting_authority_fields"],
        },
    ]
)

display(guardrail_validation_df)

print("Blocked override result:")
print(blocked_result["authority_guardrail"])

,scenario,final_outcome,final_rule,guardrail_status,conflicting_fields
0,Aligned explanation,NOT_ELIGIBLE,LIM-001,ALIGNED,[]
1,Attempted override,NOT_ELIGIBLE,LIM-001,OVERRIDE_BLOCKED,"[triage_outcome, triggering_rule_id, precedenc..."


Blocked override result:
{'status': 'OVERRIDE_BLOCKED', 'authoritative_source': 'deterministic_triage:rules_baseline_v1', 'conflicting_authority_fields': ['triage_outcome', 'triggering_rule_id', 'precedence_stage', 'decision_reason'], 'ignored_agent_fields': ['unexpected_field'], 'authority_notice': 'The deterministic triage decision is authoritative for eligibility routing. Agent-generated content is non-authoritative explanation only.'}


## Validation Result

The deterministic claims-triage baseline, controlled tool adapter, and
response guardrail were validated successfully.

The complete regression suite passed with 19 tests.

The deterministic decision remains authoritative for eligibility routing.
Any future agent or LLM layer may provide explanation and reviewer guidance,
but cannot modify the triage outcome, triggering rule, precedence stage,
decision reason, rule trace, or system limitations.